# ✂️ AutoClip – gratis di server Google
Ubah video panjang jadi klip pendek vertikal (Shorts / TikTok / Reels) otomatis. Proses berjalan di **server Google**, bukan di laptop/HP Anda.

**Cara pakai:**
1. Menu **Runtime → Ubah jenis runtime → GPU T4 → Simpan** (sekali saja).
2. Isi link video di kotak **Buat klip** di bawah, lalu klik tombol ▶ di kirinya.
3. Tunggu sampai selesai – hasil (video, thumbnail, judul & deskripsi) **otomatis terunduh** sebagai file ZIP.

Tidak perlu Google Drive. ⚠️ Potong hanya video milik sendiri, yang sudah diizinkan, atau sesuai aturan campaign.

In [ ]:
#@title ▶ Buat klip
#@markdown ### 1. Video sumber
#@markdown Link YouTube / TikTok / Twitch (beberapa video: pisahkan dengan koma)
SUMBER = ""  #@param {type:"string"}
#@markdown Atau centang ini untuk memilih file video dari laptop/HP:
VIDEO_DARI_PERANGKAT = False  #@param {type:"boolean"}
#@markdown ### 2. Klip
JUMLAH_KLIP = 5  #@param {type:"slider", min:1, max:20, step:1}
DURASI_DETIK = 60  #@param {type:"slider", min:15, max:180, step:5}
JENIS_KONTEN = "otomatis"  #@param ["otomatis", "gaming", "podcast", "reaction", "komedi", "hiburan", "vlog", "masak", "tutorial", "tutorial_pc", "interview", "fitness", "kendaraan", "unboxing", "review", "musik", "sport", "presentasi", "berita", "action", "ekstrem", "balapan", "anime", "horor", "sinematik"]
MOMEN = "otomatis"  #@param ["otomatis", "lucu", "action", "emosional", "informatif"]
#@markdown ### 3. Tampilan
NAMA_AKUN = ""  #@param {type:"string"}
#@markdown ↑ watermark kecil di video, mis. `@akunmu` (kosong = tanpa)
BAHASA_VIDEO = "id"  #@param ["id", "en", "auto"]
WARNA_SUBTITLE = "otomatis"  #@param ["otomatis", "kuning", "hijau", "merah", "oranye", "biru", "pink", "ungu", "putih"]
AJAKAN_AKHIR = "FOLLOW UNTUK PART SELANJUTNYA!"  #@param {type:"string"}
#@markdown ### 4. Pilihan lain (boleh dilewati)
#@markdown API key Google Gemini untuk thumbnail AI (aistudio.google.com → Get API key)
API_KEY_GEMINI = ""  #@param {type:"string"}
#@markdown API key Claude untuk judul & pemilihan momen lebih pintar (console.anthropic.com)
API_KEY_CLAUDE = ""  #@param {type:"string"}
#@markdown Simpan juga ke Google Drive (butuh ruang kosong di Drive)
SIMPAN_KE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
#@markdown Centang bila YouTube menolak unduhan – lalu pilih file `cookies.txt` YouTube Anda
PAKAI_COOKIES = False  #@param {type:"boolean"}

import os, shutil, subprocess, glob
if not os.path.exists('/content/AutoClip/colab_run.py'):
    REPO = "https://github.com/zuhri74/autoclip"
    import os, subprocess
    APP = '/content/AutoClip'
    print('📦 Menyiapkan AutoClip (±2–4 menit, sekali per sesi)...')
    if os.path.exists(APP + '/.git'):
        subprocess.run(['git', '-C', APP, 'pull', '-q'], check=False)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, APP], check=True)
    subprocess.run('pip -q install -U -r ' + APP + '/requirements.txt', shell=True, check=True)
    subprocess.run('mkdir -p ~/.fonts && cp -f ' + APP + '/fonts/*.ttf ~/.fonts/ 2>/dev/null; fc-cache -f', shell=True)
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
    print('✅ Siap!', ('GPU: ' + gpu) if gpu else '⚠️ GPU belum aktif – proses tetap jalan tapi lebih lambat (Runtime → Ubah jenis runtime → GPU T4).')
APP = '/content/AutoClip'
os.chdir(APP)
from google.colab import files
src = [s.strip().strip('"') for s in SUMBER.replace('\n', ',').split(',') if s.strip()]
if VIDEO_DARI_PERANGKAT:
    print('📁 Pilih file video (tunggu sampai upload 100%)...')
    os.makedirs('/content/video', exist_ok=True)
    for name, data in files.upload().items():
        p = os.path.join('/content/video', os.path.basename(name))
        with open(p, 'wb') as f:
            f.write(data)
        src.append(p)
if not src:
    raise SystemExit('❌ Isi dulu SUMBER dengan link video, atau centang VIDEO_DARI_PERANGKAT.')
if PAKAI_COOKIES:
    print('🍪 Pilih file cookies.txt...')
    for name, data in files.upload().items():
        with open(APP + '/cookies.txt', 'wb') as f:
            f.write(data)
out = '/content/hasil'
if SIMPAN_KE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    out = '/content/drive/MyDrive/AutoClip/hasil'
os.makedirs(out, exist_ok=True)
sets = ['n_clips=%d' % JUMLAH_KLIP, 'duration=%d' % DURASI_DETIK, 'genre=' + JENIS_KONTEN, 'focus=' + MOMEN,
        'handle=' + NAMA_AKUN.strip(), 'language=' + BAHASA_VIDEO, 'text_color=' + WARNA_SUBTITLE,
        'cta=' + AJAKAN_AKHIR.strip()]
if API_KEY_CLAUDE.strip():
    sets += ['ai=claude', 'api_key=' + API_KEY_CLAUDE.strip()]
env = dict(os.environ)
if API_KEY_GEMINI.strip():
    env['GEMINI_API_KEY'] = API_KEY_GEMINI.strip()
cmd = ['python', '-u', 'colab_run.py', '--out', out] + sum([['--set', x] for x in sets], []) + src
LAST_RUN = None
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in p.stdout:
    print(line, end='')
    if line.startswith('Hasil ke : '):
        LAST_RUN = line.split(':', 1)[1].strip()
p.wait()
if p.returncode != 0 or not LAST_RUN:
    raise SystemExit('\n❌ Ada masalah – lihat pesan di atas. YouTube menolak? Centang PAKAI_COOKIES atau pakai VIDEO_DARI_PERANGKAT.')
print('\n📦 Membuat ZIP & mengunduh ke perangkat Anda...')
z = shutil.make_archive('/content/AutoClip_' + os.path.basename(LAST_RUN), 'zip', LAST_RUN)
files.download(z)
print('✅ Selesai! File ZIP sedang diunduh. Jalankan kotak di bawah untuk melihat judul & thumbnail.')


In [ ]:
#@title 👀 Lihat judul, deskripsi & thumbnail hasil terakhir
import glob, os, base64, html
from IPython.display import display, HTML
runs = sorted(glob.glob('/content/hasil/*/') + glob.glob('/content/drive/MyDrive/AutoClip/hasil/*/'), key=os.path.getmtime)
if not runs:
    raise SystemExit('Belum ada hasil – jalankan kotak "Buat klip" dulu.')
last = runs[-1]
for txt in sorted(glob.glob(last + '**/*_youtube.txt', recursive=True)):
    base = txt[:-len('_youtube.txt')]
    body = open(txt, encoding='utf-8').read()
    judul = body.split('=== DESKRIPSI ===')[0].split('\n', 1)[-1].strip()
    desk = body.split('=== DESKRIPSI ===')[-1].split('=== BAHASA LAIN')[0].strip()
    th = base + '_thumbnail.jpg'
    img = ('<img src="data:image/jpeg;base64,' + base64.b64encode(open(th, 'rb').read()).decode() +
           '" style="width:360px;border-radius:10px">') if os.path.exists(th) else ''
    display(HTML('<div style="display:flex;gap:16px;margin:14px 0;font-family:sans-serif;flex-wrap:wrap">' + img +
                 '<div><b style="font-size:16px">' + html.escape(judul) + '</b><pre style="white-space:pre-wrap;max-width:560px">' +
                 html.escape(desk) + '</pre></div></div>'))
